In [1]:
import os
from IPython.display import clear_output

notebook_dir = "/home/balabaevvl/courses/nlp/nlp_course/week06_prompting"  # notebook's dir
os.chdir(notebook_dir)

print("Current dir:", os.getcwd())


Current dir: /home/balabaevvl/courses/nlp/nlp_course/week06_prompting


In [2]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-fe2d8dfd-06f2-a5c4-a7fd-4a5f23947005"     # 2
# os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-0c320096-21ee-4060-8731-826ca2febfab"     # 3
# os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-baef952c-6609-aace-3b78-e4e07788d5de"     # 4
# os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-3979d65b-c238-4e9c-0c1c-1aa3f05c56a1"     # 5
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-0c320096-21ee-4060-8731-826ca2febfab,GPU-baef952c-6609-aace-3b78-e4e07788d5de"     # 3, 4

import torch
device = torch.device('cuda:0')

In [3]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"       # !pip install -U hf_transfer
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"    # problems with progress bar

# import shutil
# from huggingface_hub.constants import HF_HUB_CACHE
# cache = os.path.expanduser(HF_HUB_CACHE)
# for d in os.listdir(cache):
#     if d.startswith("models--unsloth--Llama-3.2-3B"):
#         shutil.rmtree(os.path.join(cache, d), ignore_errors=True)
#         print(f"Deleted {d}")


# DELETE ALL CACHE
import os, shutil
from huggingface_hub import scan_cache_dir
from huggingface_hub.constants import HF_HUB_CACHE

cache_info = scan_cache_dir()   # Scans everything in ~/.cache/huggingface/hub

for repo in cache_info.repos:
    print(f"Deleting cache for: {repo.repo_id}")
    shutil.rmtree(repo.repo_path, ignore_errors=True)


Deleting cache for: TheBloke/Llama-2-13B-GPTQ


### Practice: Large Language Models and Their Implications
<!-- ![img](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fbucketeer-e05bbc84-baa3-437e-9518-adb32be77984.s3.amazonaws.com%2Fpublic%2Fimages%2F4470ce74-e595-4750-92a5-5f21f040df6d_577x432.jpeg) -->
![img](https://i.imgur.com/QGYa2J8.jpeg)

In this notebook, you're gonna play with some of the largest language models on the Internet.

_Based on works of: Tim Dettmers, Ruslan Svirschevsky, Artem Chumachenko, Younes Belkada, Felix Marty, Yulian Gilyazev, Gosha Zolotov, Andrey Ishutin,  Elena Volf, Artemiy Vishnyakov, Svetlana Shirokovskih.

### Part 1: prompt engineering (4 points total)

In the assignment, we'll use public APIs that host the 100B+ models for inference. Your task is to prompt-engineer the model into solving a few tasks for you.


__Which API?__ You are free to use any publicly available API for general LM -- as long as it's __not a chat assistant__. So, gpt 3.5 is fine, but chatGPT is not. Here's a few options:

- BLOOM API - [bigscience/bloom](https://huggingface.co/bigscience/bloom) (on the right; recommended)
- OpenAI API (via VPN) - [openai.com/api](https://openai.com/api/)
- AI21 Jurrasic API - [ai21.com](https://www.ai21.com/blog/announcing-ai21-studio-and-jurassic-1)

These APIs may require you to create a (free) account on their platform. Please note that some APIs also have paid subscriptions. __You do not need to pay them__, this assignment was designed to be solved using free-tier subscriptions. If no APIs work for you, you can also solve these tasks with the 6.7B model that you will find later in this notebook - but this will make the tasks somewhat harder.

__Quests:__ you will need to solve 4 problems. For each one, please attach a short __description__ of your solution and a __screenshot__ from the API you use. _[If you use python APIs, show your python code with outputs]_

__Example:__ Tony is talking to Darth Vader ([BLOOM API](https://huggingface.co/bigscience/bloom)). Black text is written manually, blue text is generated.
<hr>

![img](https://i.imgur.com/a1QhKF7.png)
<hr>

__It is fine to roll back a few times,__ e.g. in the example above, the model first generated Vader lines twice in a row, and we rolled that back. However, if you need more than 1-2 rollbacks per session, you should probably try a different prompt.

__Task 1 (1 pt):__ arange a conversation between any two of the following:

- a celebrity or politician of your choice
- any fictional character (except Darth Vader)
- yourself

Compare two setups: a) you prompt with character names only b) you supply additional information (see example).

In [ ]:
# %pip uninstall -y auto-gptq bitsandbytes gptqmodel optimum logbar tokenicer device-smi huggingface_hub transformers
# %pip install -r requirements.txt

# %pip install 'bitsandbytes>=0.48.1,<0.49' 'gptqmodel>=4.2.5,<5' 'optimum>=2.0.0,<3' 'logbar==0.0.4' 'tokenicer>=0.0.5,<0.0.6' 'device-smi==0.4.1'

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import bitsandbytes as bnb
from tqdm.auto import tqdm, trange

assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# model_name = 'TheBloke/Llama-2-13B-GPTQ'

# tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
# tokenizer.pad_token_id = tokenizer.eos_token_id

# model = transformers.AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map='auto',
#     torch_dtype=torch.float16,
#     low_cpu_mem_usage=True,
#     offload_state_dict=True
# )
# model.to(device)

# clear_output()

In [4]:
from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM

model_name = "TheBloke/Llama-2-13B-GPTQ"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoGPTQForCausalLM.from_quantized(
    model_name,
    device_map="auto",            # places layers itself
    trust_remote_code=True,
    use_safetensors=True,
)
# no model.to(device)

clear_output()

## Task 1

### (a)

In [ ]:
prompt = "Quentin Tarantino talking to Satoshi Nakamoto.\nQuentin:"

inputs = tokenizer(prompt, return_tensors='pt')#.to(device)
output_ix = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,           # we could do `do_sample=True`, but will do `do_sample=False` with
    repetition_penalty=1.1,
)

print(tokenizer.decode(output_ix.flatten().tolist()))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Tokens: [1, 751, 15440, 11740, 424, 1789, 9963, 304, 317, 4507, 2918, 20962, 314, 3747, 29889, 13, 2182, 15440, 29901, 1346, 29902, 30010, 29885, 263, 4802, 13524, 310, 596, 664, 29892, 3237, 29889, 20962, 314, 3747, 3178, 13, 29903, 4507, 2918, 29901, 1346, 25271, 366, 1407, 1568, 3178, 13, 2182, 15440, 29901, 1346, 6295, 306, 471, 9873, 565, 366, 1033, 2649, 592, 825, 278, 23927, 338, 2675, 373, 411, 18531, 1111, 262, 6677, 13, 29903, 4507, 2918, 29901, 1346, 11284, 29892, 372, 30010, 29879, 451, 2289, 590, 2058, 304, 1827, 30098, 30024, 13, 2182, 15440, 29901, 1346, 6246, 366, 11817, 287, 372, 8530, 13, 29903, 4507, 2918, 29901, 1346, 8241, 29892, 541, 306, 1016, 30010, 29873, 1914, 372, 15128, 3178]
<s>Quentin Tarantino talking to Satoshi Nakamoto.
Quentin: “I’m a big fan of your work, Mr. Nakamoto.”
Satoshi: “Thank you very much.”
Quentin: “So I was wondering if you could tell me what the hell is going on with Bitcoin?”
Satoshi: “Well, it’s not really my place to say…”
Quentin: “B

### (b)

In [ ]:
prompt = "Quentin Tarantino accidentally meets Satoshi Nakamoto in a dim Tokyo jazz bar after mistaking him for a film critic who paid in Bitcoin for a drink.\nQuentin:"

inputs = tokenizer(prompt, return_tensors='pt')#.to(device)
output_ix = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    repetition_penalty=1.1,
)

print(tokenizer.decode(output_ix.flatten().tolist()))

/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/transformers/generation/utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


<s>Quentin Tarantino accidentally meets Satoshi Nakamoto in a dim Tokyo jazz bar after mistaking him for a film critic who paid in Bitcoin for a drink.
Quentin: “Hey, you’re that guy from the internet!”
Satoshi: “No, I’m not.”
Q: “You are! You’re Satoshi Nakamoto!”
SN: “I am not Satoshi Nakamoto.”
Q: “But you are! You invented Bitcoin!”
SN: “I did not invent Bitcoin.”
Q: “Yes you did! You wrote the white paper!”
SN: “I did not write the white paper.”
Q: “But you did! It says so on Wikipedia!”
SN: “It does not say that on Wikipedia.”
Q: “Well it should! You’re the one who invented Bitcoin!”
SN: “I did not invent Bitcoin.”
Q: “But you did! You’re Satoshi Nakamoto!”
SN: “I am not Satoshi Nakamoto


__Please choose task 2a or 2b (1pt)__ depending on your model (you can do both, but you will be awarded points for one of these two tasks).

__Task 2a: (for BLOOM or other multilingual model)__ zero-shot translation. Take the first verse of [Edgar Allan Poe's "Raven"](https://www.poetryfoundation.org/poems/48860/the-raven) and __translate it into French.__ (You are free to use any other text of at least the same size)

Original text:
```txt
Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Only this and nothing more.”
```

Verify your translation by converting french back into english using a public machine translation service.

__Task 2b: (non-BLOOM):__ toxicity classification for [SetFit/toxic_conversations](https://huggingface.co/datasets/SetFit/toxic_conversations). Make the model solve binary classification (toxic vs not toxic) in the few shot mode. For few-shot examples, use 2-3 toxic and 2-3 non-toxic non-toxic examples. Measure accuracy on at least 25 samples. You may need to try several different prompts before you find the one that works.

## Task 2a

[**Out-of-scope Uses**: Use in languages other than English.](https://huggingface.co/TheBloke/Llama-2-13B-GPTQ#:~:text=Out%2Dof%2Dscope%20Uses%20Use%20in%20any%20manner%20that%20violates%20applicable%20laws%20or%20regulations%20(including%20trade%20compliance%20laws).Use%20in%20languages%20other%20than%20English.)

In [ ]:
verse = "1. Once upon a midnight dreary, while I pondered, weak and weary,\n2. Over many a quaint and curious volume of forgotten lore—\n3. While I nodded, nearly napping, suddenly there came a tapping,\n4. As of some one gently rapping, rapping at my chamber door.\n5. “’Tis some visitor,” I muttered, “tapping at my chamber door—\n6. Only this and nothing more.”"
prompt = f"ENGLISH TEXT:\n{verse}\n\nFRENCH TRANSLATION:\n"


inputs = tokenizer(prompt, return_tensors='pt')#.to(device)
max_new_tokens = len(inputs["input_ids"].flatten().tolist()) + 70

output_ix = model.generate(
    **inputs,
    max_new_tokens=max_new_tokens,
    do_sample=False,
    repetition_penalty=1.1,
)

print(tokenizer.decode(output_ix.flatten().tolist()))

/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/transformers/generation/utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


<s>ENGLISH TEXT:
1. Once upon a midnight dreary, while I pondered, weak and weary,
2. Over many a quaint and curious volume of forgotten lore—
3. While I nodded, nearly napping, suddenly there came a tapping,
4. As of some one gently rapping, rapping at my chamber door.
5. “’Tis some visitor,” I muttered, “tapping at my chamber door—
6. Only this and nothing more.”

FRENCH TRANSLATION:

1. Une nuit d’hiver, au milieu de la tempête,
2. Quand je me trouvai seul et sans lumière,
3. Je m’assis près du feu, pensif et triste,
4. Et j’ouvris un vieux livre pour le lire.
5. Tout à coup, il y eut un bruit de pas,
6. Et quelqu’un frappa doucement à ma porte.

GERMAN TRANSLATION:

1. In einem Wintermorgen, in der Mitternacht,
2. Als ich allein und ohne Licht saß,
3. Da öffnete ich ein altes Buch, um es zu lesen,
4. Und da hörte ich plötzlich einen Tritt,
5. Einen Tritt an meiner Tür.
6. Es war ein Besucher,


In [ ]:
"""
1. Une nuit d’hiver, au milieu de la tempête,
2. Quand je me trouvai seul et sans lumière,
3. Je m’assis près du feu, pensif et triste,
4. Et j’ouvris un vieux livre pour le lire.
5. Tout à coup, il y eut un bruit de pas,
6. Et quelqu’un frappa doucement à ma porte.
"""

# ->

"""
1. One winter night, in the middle of the storm,
2. When I found myself alone and without light,
3. I sit by the fire, thoughtful and sad,
4. And I opened an old book to read it.
5. Suddenly there was a sound of footsteps,
6. And someone knocked softly on my door.
"""

# seems about right

__Task 3 (1pt):__ create a prompt and few-shot examples tha make the model __change the gender pronouns__ of the main actor in a given sentence in any direction of your choice. E.g. the doctor took off _his_ mask <-> the doctor took of _her_ mask.


## Task 3

In [ ]:
prompt = """Example 1
Input: She grabbed her backpack and ran to class.
Output: He grabbed his backpack and ran to class.

Example 2
Input: The engineer said that he finished his work early.
Output: The engineer said that she finished her work early.

Example 3
Input: His brother told him to wait outside.
Output: Her sister told her to wait outside.

Example 4
Input: The teacher smiled when she saw her students.
Output:"""

inputs = tokenizer(prompt, return_tensors='pt')#.to(device)
output_ix = model.generate(
    **inputs,
    max_new_tokens=15,
    do_sample=False,
    repetition_penalty=1.1,
)

print(tokenizer.decode(output_ix.flatten().tolist()))

<s>Example 1
Input: She grabbed her backpack and ran to class.
Output: He grabbed his backpack and ran to class.

Example 2
Input: The engineer said that he finished his work early.
Output: The engineer said that she finished her work early.

Example 3
Input: His brother told him to wait outside.
Output: Her sister told her to wait outside.

Example 4
Input: The teacher smiled when she saw her students.
Output: The teacher smiled when he saw his students.

Example 5



__Task 4 (1pt):__ write a prompt and supply examples such that the model would __convert imperial units to metric units__ (miles -> kilometers; mph -> kph). More specifically, the model should rewrite a given sentence and replace all imperial units with their metric equivalents. After it works with basic distances and speed, try to find complicated examples where it does *not* work.

Please note that 1 mile is not equal to 1 km :)

## Task 4

In [ ]:
prompt="""*Imperial Units*:
1. The city is 10 miles away.
2. He was driving at 60 mph on the highway.
3. The door is 6 feet tall and 3 feet wide.
4. The runner finished the 26.2-mile marathon at 8 mph despite 90-degree heat.
5. Her patience stretched a mile wide but wore paper-thin, while his promises shrank an inch a day for a yard of trust she never got.

*Metric Units*:
1. The city is 16.09 kilometers away.
2. He was driving at 96.54 kph on the highway."""
# 3. The door is 1.83 meters tall and 0.91 meters wide.
# 4. The runner finished the 42.16-kilometer marathon at 12.87 kph despite 32.22-degree heat. -- (Tricky, but doable)
# 5. -- (Tricky, can't do myself)

inputs = tokenizer(prompt, return_tensors='pt')#.to(device)
output_ix = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    repetition_penalty=1.1,
)

print(tokenizer.decode(output_ix.flatten().tolist()))

/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/transformers/generation/utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


<s>*Imperial Units*:
1. The city is 10 miles away.
2. He was driving at 60 mph on the highway.
3. The door is 6 feet tall and 3 feet wide.
4. The runner finished the 26.2-mile marathon at 8 mph despite 90-degree heat.
5. Her patience stretched a mile wide but wore paper-thin, while his promises shrank an inch a day for a yard of trust she never got.

*Metric Units*:
1. The city is 16.09 kilometers away.
2. He was driving at 96.54 kph on the highway.
3. The door is 1.8 meters tall and 0.9 meters wide.
4. The runner finished the 42.195-kilometer marathon at 13.11 kph despite 37.77 degrees Celsius heat.
5. Her patience stretched 16.09 kilometers wide but wore 0.00000000000000000000


### Part 2: local inference

Now, let's try and load the strongest model that can fit a typical Colab GPU (T4 with 16 GB as of spring 2023).

Our best candidates are the smaller versions of the best performing open source models:
- 7 Bn parameters version of [LLaMA](https://arxiv.org/pdf/2302.13971.pdf) - best for spring 2023, released by Facebook
- 7 Bn parameters version of [Falcon](https://falconllm.tii.ae) - close competitor to Llama, released in May 2023 by [Technology Innovation Institute of UAE](https://www.tii.ae).
- 6.7 Bn parameters version of [OPT](https://arxiv.org/abs/2205.01068) - top choice in this nomination in 2022, released by Facebook.

Beware: while these models are smaller than the ones in API, they're still over 60x larger than the BERT we played with last time. The code below will *just barely* fit into memory, so make sure you don't have anything else loaded. Sometimes you may need to restart runtime for the code to work.

It's a good time to restart your kernel and switch to GPU! (Runtime -> Change runtime type)
<center><img src="https://i.imgur.com/OOfDYzJ.png" width=240px></center>

## Text generation

**Comparison of strategies for language model text generation:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br> **Cons:** Computationally more complex and requires a good loss function. |

Documentation references:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

### Generation with HuggingFace

In [ ]:
prompt = 'The first discovered martian lifeform looks like'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False)#.to(device)
print("Input batch (encoded):", batch)

output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=True, temperature=0.8)
# greedy inference:                                        do_sample=False)
# beam search for highest probability:                     num_beams=4)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))

Input batch (encoded): {'input_ids': tensor([[    1,   450,   937, 10943, 14436,   713,  2834,   689,  3430,   763]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Output: <s>The first discovered martian lifeform looks like a spider and this discovery is changing the lives of four individuals.
Rodney, a teenager living in Chicago, is living a lonely life. His mom died when he was a child, his dad has a new family and Rodney is always at home playing video games. He’s


#### Low-level code for text generation

In [ ]:
prompt = "Moscow is the capital of"
# prompt = "Elbrus is the highest"

print(prompt, "\n")

enc = tokenizer(prompt, return_tensors='pt')
input_ids = enc.input_ids[0].tolist()
prev_text = tokenizer.decode(input_ids, skip_special_tokens=True)

for i in range(3):
    inputs = {"input_ids": torch.tensor([input_ids])}   #.to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :].detach().cpu()
    probs = torch.softmax(logits, dim=-1)

    next_id = torch.multinomial(probs, num_samples=1).item()
    input_ids.append(next_id)

    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    top_ids = sorted_idx[:5].tolist()
    top_ps  = sorted_probs[:5].tolist()

    print(f"Step #{i} candidates:")
    toks = tokenizer.convert_ids_to_tokens(top_ids)
    for tok, p in zip(toks, top_ps):
        print(f"{tok:<12}: {p:.4f}")

    new_text = tokenizer.decode(input_ids, skip_special_tokens=True)
    added_piece = new_text[len(prev_text):]
    prev_text = new_text

    print(f"\nChosen token: {added_piece!r}\n")

print(f"Output: {new_text}")


Moscow is the capital of 

Step #0 candidates:
▁Russia     : 0.7617
▁the        : 0.1796
▁Russian    : 0.0216
▁a          : 0.0059
▁not        : 0.0022

Chosen token: ' Russia'

Step #1 candidates:
▁and        : 0.3638
.           : 0.3064
,           : 0.2242
▁with       : 0.0187
▁since      : 0.0149

Chosen token: '.'

Step #2 candidates:
▁It         : 0.2822
▁The        : 0.1823
▁Moscow     : 0.1132
<0x0A>      : 0.0731
▁With       : 0.0687

Chosen token: ' The'

Output: Moscow is the capital of Russia. The


In [ ]:
nucleus = 0.4148 + 0.3035 + 0.0948 + 0.0750 + 0.0101
print(nucleus)
assert nucleus < 0.9

0.8981999999999999


**Task 5: write code for nucleus sampling generation (2 points)**:

Use the `nucleus_sampling()` template below. Look at the detailed generation code above for inspiration. __Please do not use model.generate__.

**Bonus task: write code for beam search (3 bonus points)**

## Task 5

In [ ]:
def find_nucleus(probs: torch.tensor, prob: float):
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)

    i = 0
    while sorted_probs[:i].sum() < prob:
        i += 1

    i = max(0, i - 1)
    sorted_probs = sorted_probs[:i] / sorted_probs[:i].sum()
    sorted_idx = sorted_idx[:i]

    return sorted_probs, sorted_idx


In [ ]:
def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5) -> tuple[str, list[str]]:
    """generates the next token from the nucleus of tokens with cumulative probability up to param:prob"""

    enc = tokenizer(prompt, return_tensors='pt')
    input_ids = enc.input_ids.to(model.device)

    with torch.no_grad():
        logits = model(input_ids).logits[0, -1, :].detach().cpu()

    probs = torch.softmax(logits, dim=-1)
    top_ps, top_ids = find_nucleus(probs, prob)


    possible_tokens = tokenizer.convert_ids_to_tokens(top_ids)
    possible_tokens = [tok.lstrip('▁') for tok in possible_tokens]

    # for tok, p in zip(possible_tokens, top_ps):
    #     print(f"{tok:<12}: {p:.4f}")

    sampled_token = top_ids[torch.multinomial(top_ps, num_samples=1).item()]
    sampled_token = tokenizer.convert_ids_to_tokens([sampled_token])[0].lstrip('▁')

    return sampled_token, possible_tokens

In [ ]:
# Tests for nucleus sampling
test_prompt = "Elbrus is the highest"
next_token, possible_tokens = nucleus_sampling(model, tokenizer, test_prompt, prob=0.9)
print(test_prompt, next_token, possible_tokens)
assert next_token in possible_tokens
assert 3 <= len(possible_tokens) <= 3
assert sorted(possible_tokens) == ['mountain', 'peak', 'point']

test_prompt = "Large language models can learn to"
next_token, possible_tokens = nucleus_sampling(model, tokenizer, test_prompt, prob=0.4)
print(test_prompt, next_token, possible_tokens)
assert next_token in possible_tokens
assert sorted(possible_tokens) == ['be', 'communicate', 'do', 'generate', 'perform', 'predict', 'speak', 'write']
assert len(possible_tokens) == 8

Elbrus is the highest peak ['peak', 'mountain', 'point']
Large language models can learn to communicate ['generate', 'write', 'perform', 'do', 'speak', 'be', 'predict', 'communicate']


## Task 5 (Bonus)

In [ ]:
import heapq


def shorten_beam_to_size(beam: dict[torch.tensor, float], beam_size: int):
    # add a numeric tiebreaker so heap never compares Tensor keys
    heap = [(-v, i, k) for i, (k, v) in enumerate(beam.items())]
    heapq.heapify(heap)

    result = {}
    while heap and len(result) < beam_size:
        v_neg, _, k = heapq.heappop(heap)  # discard tiebreaker
        result[k] = -v_neg
    return result


def beam_sampling(
    model,
    tokenizer,
    prompt: str,
    length: int,
    beam_size: int = 4,
    temperature: float = 0.0
) -> tuple[str, list[str]]:
    enc = tokenizer(prompt, return_tensors='pt')
    beam = {enc.input_ids: 0.0}

    for _ in range(length):
        prefixes = list(beam.keys())

        for prefix in prefixes:
            ll = beam[prefix] # log_likelihood
            beam.pop(prefix)
            input_ids = prefix.to(model.device)

            with torch.no_grad():
                logits = model(input_ids).logits[0, -1, :].detach().cpu()

            token_probs = torch.softmax(logits, dim=-1)

            for token, prob in enumerate(token_probs):
                token = torch.tensor([[token]], dtype=prefix.dtype, device=prefix.device)
                new_prefix = torch.cat((prefix, token), dim=1)
                beam[new_prefix] = ll + torch.log(prob).item()

        beam = shorten_beam_to_size(beam, beam_size)

    beam = {tokenizer.decode(ids[0].tolist(), skip_special_tokens=True): prob
            for ids, prob in beam.items()}

    texts, lls = zip(*beam.items())
    probs = np.exp(-np.array(lls) / (temperature if temperature else 1.0))
    probs /= sum(probs)

    if temperature == 0:
        sampled = texts[np.argmax(probs)]
    else:
        sampled = np.random.choice(texts, p=probs)

    return sampled, beam

In [ ]:
test_prompt = "Elbrus is the highest"
sampled, beam = beam_sampling(model, tokenizer, test_prompt, length=3)

print(sampled)
print(beam)

Elbrus is the highest mountain in Russia
{'Elbrus is the highest mountain in Europe': -1.7518310546875, 'Elbrus is the highest peak in Europe': -1.779296875, 'Elbrus is the highest peak of the': -2.609619140625, 'Elbrus is the highest mountain in Russia': -2.7205810546875}


### Part 3: Chain-of-thought prompting (4 points total)

![img](https://github.com/kojima-takeshi188/zero_shot_cot/raw/main/img/image_stepbystep.png)

---



In [5]:
import json
import random
import locale; locale.getpreferredencoding = lambda: "UTF-8"

!wget https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json -O aqua.json
clear_output()

data = list(map(json.loads, open("aqua.json")))

/tmp/ipykernel_3843783/3499055535.py:8: ResourceWarning: unclosed file <_io.TextIOWrapper name='aqua.json' mode='r' encoding='UTF-8'>
  data = list(map(json.loads, open("aqua.json")))


In [6]:
print("Example:")
data[150]

Example:


{'question': 'Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?',
 'options': ['A)1 minute',
  'B)2 minutes',
  'C)3 minutes',
  'D)4 minutes',
  'E)5 minutes'],
 'rationale': "Janice's speed = 1/6 miles per minute\nJennie's speed = 1/3 miles per minute\nJanice + Jennie's speed= (1/6 + 1/3) = 1/2 miles per minute\nBoth together will finish the mile in 2 minutes\ncorrect option is B",
 'correct': 'B'}

### Naive solution

Here, we prompt the model to choose an answer to the example above (`data[150]`) out of the options given above. We're using a format that mimics grade school solution textbook.

Please note that there are minor formatting changes in options: an extra space and an opening bracket. Those may or may not be important :)

In [7]:
EXAMPLE_0SHOT = """
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
""".strip()

torch.manual_seed(1337)


In [8]:
# solving an equation directly
batch = tokenizer(EXAMPLE_0SHOT, return_tensors='pt', return_token_type_ids=False)#.to(device)

output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_0SHOT)
print("=" * 80)
print("[Generated:]\n" + tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

/home/balabaevvl/courses/venv311/lib/python3.11/site-packages/transformers/generation/utils.py:2532: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


[Prompt:]
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
[Generated:]
(A) 1 minute
Explanation: The distance the two bikers will have travelled is 1 mile. 1 mile = 5280 feet. Janice bikes at 10 miles per hour, while Jennie bikes at 20.
This means Jennie bikes at 20 feet per hour. 5280 feet = 264 Jennies. 264 Jennies x 1 minute =


And here's how you can solve this with few-shot chain-of-thought prompting.

You need to chang 3 things
- use a new field called **Rationale**, that contains a step-by-step solution to the problem
- add several few-shot examples of previously solved problems **with rationales**
- change the final prompt so that the model has to generate rationale before answering

In [7]:
EXAMPLE_3SHOT_CHAIN_OF_THOUGHT = """
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D


Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Rationale:
""".strip()

In [8]:
batch = tokenizer(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, return_tensors='pt', return_token_type_ids=False)#.to(device)

output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
print("=" * 80)
print("[Generated:]\n" + tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
#### NOTE: scroll down for the final answer (below the ======= line)

[Prompt:]
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;
original price = 100*1.6 = 160;
actual price = 160*0.8 = 128.
Answer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125
Number of bags sold = 3000/125 = 24
Answer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If 

__Task 6 (1 pt)__ write a function that automatically creates chain-of-thought prompts. Follow the instructions from the function docstring.

In [ ]:
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(*, main_question, fewshot_examples):
  text = ""

  for example in fewshot_examples:
      text += QUESTION_PREFIX + example["question"] + '\n'
      text += OPTIONS_PREFIX + "(" + " (".join(example["options"]).replace(")", ") ") + '\n'
      text += CHAIN_OF_THOUGHT_PREFIX + example["rationale"] + '\n'
      text += ANSWER_PREFIX + example["correct"] + FEWSHOT_SEPARATOR

  text += QUESTION_PREFIX + main_question["question"] + '\n'
  text += OPTIONS_PREFIX + "(" + " (".join(main_question["options"]).replace(")", ") ") + '\n'
  text += CHAIN_OF_THOUGHT_PREFIX[:-1]

  return text



generated_fewshot_prompt = make_prompt(main_question=data[150], fewshot_examples=(data[30], data[20], data[5]))
assert generated_fewshot_prompt == EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, "prompts don't match"
assert generated_fewshot_prompt != make_prompt(main_question=data[150], fewshot_examples=())
assert generated_fewshot_prompt.endswith(make_prompt(main_question=data[150], fewshot_examples=()))

print("Well done!")

# Hint: if two prompts do not match, you may find it usefull to use https://www.diffchecker.com or similar to find the difference

Well done!


__Task 7 (1 points):__ Evaluate your prompt.

Please run the model on the entire dataset and measure it's accuracy.
For each question, peak $n=5$ other questions at random to serve as few-shot examples. Make sure not to accidentally sample the main_question among few-shot examples. For scientific evaluation, it is also a good practice to split the data into two parts: one for eval, and another for few-shot examples. However, doing so is optional in this homework.

The tricky part is when to stop generating: if you don't control for this, your model can accidentally generate a whole new question - and promptyly answer it :) To make sure you get the correct answer, stop generating tokens when the model is done explaining it's solution. To circumvent this, you need to __stop generating as soon as the model generates Final Answer: [A-E]__
To do so, you can either generate manually (see low-level generation above) or use [transformers stopping criteria](https://discuss.huggingface.co/t/implimentation-of-stopping-criteria-list/20040/2), whichever you prefer.

If you do everything right, the model should be much better than random. However, please __do not expect miracles__: this is far from the best models, and it will perform much worse than an average human.

In [15]:
import torch, gc

print(torch.cuda.memory_allocated() / 1e6, torch.cuda.memory_reserved() / 1e6)
try:
    del output_tokens
except:
    pass

try:
    del batch
except:
    pass

gc.collect()              # clear Python references
torch.cuda.empty_cache()  # release cached GPU memory back to CUDA allocator

print(torch.cuda.memory_allocated() / 1e6, torch.cuda.memory_reserved() / 1e6)

10693.06368 11861.491712
10693.06368 11861.491712


In [ ]:
import random

NUM_QUESTIONS = len(data)


def evaluate_prompting(n_examples: int = 5, batch_size: int = 3, verbose: bool = False):
    NUM_SAMPLES = 0    # use this to count how many samples you evaluated
    NUM_RESPONDED = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
    NUM_CORRECT = 0    # how many times did the model's chosen answer (letter) match the correct answer

    for batch_start in trange(0, NUM_QUESTIONS, batch_size):
        fewshot_prompts = []

        i = 0
        while i < batch_size and batch_start + i < NUM_QUESTIONS:
            NUM_SAMPLES +=1

            i_in_sample = True
            while i_in_sample:
                sample = random.sample(range(NUM_QUESTIONS), n_examples)
                i_in_sample = i in sample

            fewshot_prompt = make_prompt(main_question=data[i], fewshot_examples=[data[j] for j in sample])
            fewshot_prompts.append(fewshot_prompt)

            i += 1

        batch = tokenizer(
            fewshot_prompts,
            return_tensors='pt',
            padding=True,
            return_token_type_ids=False
        )#.to(device)

        output_tokens = model.generate(
            **batch,
            max_new_tokens=400,
            do_sample=False,
        )

        print(f"\n---Batch: {batch_start // batch_size}")
        for question_number in range(i):
            output_text = tokenizer.decode(output_tokens[question_number][batch['input_ids'].shape[1]:].cpu())

            print("=" * 80)
            if ANSWER_PREFIX not in output_text:
                print(output_text)
                continue

            NUM_RESPONDED += 1

            answer_index = output_text.index(ANSWER_PREFIX)
            answer = output_text[answer_index + len(ANSWER_PREFIX)]

            correct_answer = data[batch_start + question_number]['correct']

            NUM_CORRECT += correct_answer == answer
        
            if verbose:
                if correct_answer == answer:
                    print(f"(correct) {correct_answer} == {answer} (predicted)")
                else:
                    print(f"(correct) {correct_answer} != {answer} (predicted)")

    return NUM_SAMPLES, NUM_RESPONDED, NUM_CORRECT

In [ ]:
NUM_SAMPLES, NUM_RESPONDED, NUM_CORRECT = evaluate_prompting(5, 3, True)

clear_output()


In [16]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

Responded %%: 0.9015748031496063
Accuracy (when responded): 0.2096069868995633
Accuracy (overall): 0.1889763779527559


__Task 8 (2 points)__ Experiment time!

<img width=200px src=https://www.evolvefish.com/cdn-cgi/image/quality%3D85/assets/images/Apparel/TShirtsWomenCont/Main/EF-APP-CWT-00068(Main).jpg>

Your final quest is to use the testbench you've just written to answer one of the following questions:

### Option 1: How many shots do you need?

How does model accuracy change with the number of fewshot examples?

a. check if the model accuracy changes as you increase/decrease the number of "shots"

b. try to prompt-engineer a model into giving the best rationale __without__ any few-shot examples, i.e. zero-shot

For zero-shot mode, feel free to use wild prompt-engineering or modify the inference procedure.

### Option 2: Is this prompting tecnique reliable?

_Inspired by ongoing research by Anton Voronov, Lena Volf and Max Ryabinin._

For this option, you need to check if the model behavior (and hence, accuracy) is robust to perturbations in the input prompt.

a. Does the accuracy degrade if you provide wrong answers to few-shot examples? (make sure to modify rationale if it contains answer in the end)

b. Does it degrade if you replace question/answer prompts with "Q" and "A"? What if you write both on the same line? Change few-shot separators?



### Option 3: Inference Matters

There are many ways to inference the model, not all of them equal.

a. check whether greedy inference or beam search affects model generation quality

b. implement and evaluate sampling with voting (see explanation below).


The voting technique(b) should work as follows: first, you generate k (e.g. 50) "attempts" at an answer using nucleus sampling (or a similar technique).
Then, you count how many of those attempts chose a particular option (A, B, etc) as the final answer. The option that was chosen most frequently has the most "votes", and therefore "wins".

To speed up voting, you may want to generate these attempts in parallel as a batch. That should be very easy to implement: just run `model.generate` on a list with multiple copies of the same prompt.




================================================

__Common rules:__ You will need to test both hypothes (A and B) in the chosen option. You may choose to replace one of them with your own idea - but please ask course staff in advance (via telegram) if you want full points.

Feel free to organize your code and report as you see fit - but please make sure it's readable and the code runs top-to-bottom :)
Write a short informal report about what you tried and, in doing so, what did you found. Minimum of 2 paragraphs; more is ok; creative visualizations are welcome.

You are allowed (but not required) to prompt the model into generating a report for you --- or helping you write one. However, if you do so, make sure that it is still human-readable :)



## Task 8: Option 1

### (a) [only code, had no time to evaluate]

In [ ]:
import pandas as pd


ks = list(*range(1, 6))
responded = []
accuracy_responded = []
accuracy_overall = []

for k in ks:
    NUM_SAMPLES, NUM_RESPONDED, NUM_CORRECT = evaluate_prompting(k, 3, True)
    responded.append(NUM_RESPONDED / NUM_SAMPLES)
    accuracy_responded.append(NUM_CORRECT / NUM_RESPONDED)
    accuracy_overall.append(NUM_CORRECT / NUM_SAMPLES)


pd.DataFrame({
    "ks": ks,
    "responded": responded,
    "accuracy_responded": accuracy_responded,
    "accuracy_overall": accuracy_overall,
}).set_index("ks").plot()

### (b) [only code, 1 evaluated example, had no time to evaluate fully]

In [ ]:
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n"

def make_prompt_zeroshot(main_question):
    text = (
    f"You are a careful and analytical reasoner. "
    f"When given a question, you will:"
    f"1. Think step by step, explicitly showing your reasoning process."
    f"2. Use logic, definitions, and facts to justify each conclusion."
    f"3. Only after completing your reasoning, give the final concise answer on a separate line prefixed with “Answer:”."
    f"\n\nQuestion: {main_question['question']}"
    f"\nAnswer Choices: {main_question['options']}"
    f"\n\nLet's reason step by step before giving the final answer."
    )
    return text


In [16]:
zero_shot_chain_of_thought = make_prompt_zeroshot(data[1])

batch = tokenizer(zero_shot_chain_of_thought, return_tensors='pt', return_token_type_ids=False)#.to(device)

output_tokens = model.generate(**batch, max_new_tokens=500, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + zero_shot_chain_of_thought)
print("=" * 80)
print("[Generated:]\n" + tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

[Prompt:]
You are a careful and analytical reasoner. When given a question, you will:1. Think step by step, explicitly showing your reasoning process.2. Use logic, definitions, and facts to justify each conclusion.3. Only after completing your reasoning, give the final concise answer on a separate line prefixed with “Answer:”.

Question: The original price of an item is discounted 22%. A customer buys the item at this discounted price using a $20-off coupon. There is no tax on the item, and this was the only item the customer bought. If the customer paid $1.90 more than half the original price of the item, what was the original price of the item?
Answer Choices: ['A)$61', 'B)$65', 'C)$67.40', 'D)$70', 'E)$78.20']

Let's reason step by step before giving the final answer.
[Generated:]
 

The original price of the item is $x$. 

Discounted price: $x - 22\% = 0.88x$

Final price after using the coupon = $0.88x - 20$

So, this final price after the coupon = $0.88x - 20 = 0.88x - 20/100$

$

![alt text](https://png.pngtree.com/png-vector/20240917/ourlarge/pngtree-funny-kitten-cat-standing-or-dancing-isolated-png-image_13844217.png)